# 🧪 Tox21 Dataset - Exploratory Data Analysis

This notebook explores the Tox21 dataset used for drug toxicity prediction.

**Contents:**
1. Load & overview the dataset
2. Target class distribution (toxic vs non-toxic)
3. Missing value analysis across assays
4. Molecular descriptor statistics
5. Correlation analysis
6. Feature distributions (toxic vs non-toxic)
7. Key insights & next steps

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Paths
DATA_RAW = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')

print('Setup complete ✅')

## 1. Load & Overview the Raw Dataset

In [ ]:
# Load raw Tox21 dataset
df = pd.read_csv(DATA_RAW / 'tox21.csv')
print(f'Dataset shape: {df.shape[0]} compounds × {df.shape[1]} columns')
print(f'\nColumns: {list(df.columns)}')
df.head()

In [ ]:
# Separate target columns from metadata
target_cols = [c for c in df.columns if c not in ['smiles', 'mol_id']]
print(f'Target assays ({len(target_cols)}): {target_cols}')
print(f'\nBasic statistics:')
df[target_cols].describe()

## 2. Target Class Distribution

In [ ]:
# Class distribution for each assay
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
fig.suptitle('Class Distribution Across 12 Toxicity Assays', fontsize=16, fontweight='bold')

for i, col in enumerate(target_cols):
    ax = axes[i // 4, i % 4]
    counts = df[col].value_counts().sort_index()
    colors = ['#2ecc71', '#e74c3c']  # green=non-toxic, red=toxic
    bars = ax.bar(counts.index, counts.values, color=colors[:len(counts)], edgecolor='white', linewidth=1.5)
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Non-toxic', 'Toxic'], fontsize=9)
    
    # Add count labels on bars
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                str(val), ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
summary = []
for col in target_cols:
    tested = df[col].notna().sum()
    toxic = (df[col] == 1).sum()
    summary.append({'Assay': col, 'Tested': tested, 'Toxic': toxic,
                    'Non-toxic': tested - toxic, 'Toxic %': round(toxic/tested*100, 1),
                    'Untested': len(df) - tested})
summary_df = pd.DataFrame(summary)
summary_df.style.background_gradient(subset=['Toxic %'], cmap='YlOrRd')

## 3. Missing Value Analysis

In [ ]:
# Missing values heatmap
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart of missing %
missing_pct = (df[target_cols].isnull().sum() / len(df) * 100).sort_values(ascending=True)
colors = plt.cm.YlOrRd(missing_pct / missing_pct.max())
missing_pct.plot(kind='barh', ax=ax1, color=colors, edgecolor='white')
ax1.set_title('Missing Values per Assay (%)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Missing %')
for i, (val, name) in enumerate(zip(missing_pct.values, missing_pct.index)):
    ax1.text(val + 0.3, i, f'{val:.1f}%', va='center', fontsize=9)

# Heatmap of missing pattern (sample 200 rows)
sample_idx = np.random.RandomState(42).choice(len(df), 200, replace=False)
sns.heatmap(df[target_cols].iloc[sample_idx].isnull().T, cbar=False,
            cmap='YlOrRd', ax=ax2, yticklabels=True)
ax2.set_title('Missing Pattern (200 sample compounds)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Compound index')

plt.tight_layout()
plt.savefig('../reports/missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Molecular Descriptor Analysis

Now let's look at the **computed molecular descriptors** from RDKit.

In [ ]:
# Load processed features and labels
features = pd.read_csv(DATA_PROCESSED / 'features.csv')
labels = pd.read_csv(DATA_PROCESSED / 'labels.csv')

label_col = [c for c in labels.columns if c != 'smiles'][0]
y = labels[label_col].values

print(f'Features shape: {features.shape}')
print(f'Label column: {label_col}')
print(f'Class 0 (non-toxic): {(y == 0).sum()}')
print(f'Class 1 (toxic):     {(y == 1).sum()}')
print(f'\nFirst 10 descriptor names: {list(features.columns[:10])}')
features.describe().T.head(15)

In [ ]:
# Distribution of key molecular properties
key_descriptors = ['MolWt', 'MolLogP', 'NumHDonors', 'NumHAcceptors',
                   'NumRotatableBonds', 'TPSA', 'NumAromaticRings', 'NumHeteroatoms']

# Filter to only available descriptors
key_descriptors = [d for d in key_descriptors if d in features.columns]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Distribution of Key Molecular Descriptors (Toxic vs Non-toxic)',
             fontsize=16, fontweight='bold')

for i, desc in enumerate(key_descriptors):
    ax = axes[i // 4, i % 4]
    
    # Separate by class
    non_toxic_vals = features.loc[y == 0, desc]
    toxic_vals = features.loc[y == 1, desc]
    
    ax.hist(non_toxic_vals, bins=40, alpha=0.6, color='#2ecc71', label='Non-toxic', density=True)
    ax.hist(toxic_vals, bins=40, alpha=0.6, color='#e74c3c', label='Toxic', density=True)
    ax.set_title(desc, fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../reports/descriptor_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Correlation Analysis

In [ ]:
# Correlation of top descriptors with toxicity
features_with_label = features.copy()
features_with_label['Toxic'] = y

# Get correlations with target
correlations = features_with_label.corr()['Toxic'].drop('Toxic').sort_values()

# Plot top 15 positive and top 15 negative correlations
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Most negatively correlated
top_neg = correlations.head(15)
top_neg.plot(kind='barh', ax=ax1, color='#3498db', edgecolor='white')
ax1.set_title('Top 15 Negatively Correlated with Toxicity', fontsize=13, fontweight='bold')
ax1.set_xlabel('Pearson Correlation')

# Most positively correlated
top_pos = correlations.tail(15)
top_pos.plot(kind='barh', ax=ax2, color='#e74c3c', edgecolor='white')
ax2.set_title('Top 15 Positively Correlated with Toxicity', fontsize=13, fontweight='bold')
ax2.set_xlabel('Pearson Correlation')

plt.tight_layout()
plt.savefig('../reports/correlation_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Print top 10 each direction
print('\n🔴 Top 10 features POSITIVELY correlated with toxicity:')
for name, val in correlations.tail(10).items():
    print(f'   {name:<30} r = {val:+.4f}')

print('\n🔵 Top 10 features NEGATIVELY correlated with toxicity:')
for name, val in correlations.head(10).items():
    print(f'   {name:<30} r = {val:+.4f}')

In [ ]:
# Correlation heatmap of top descriptors
top_features = list(correlations.head(10).index) + list(correlations.tail(10).index)
top_features_available = [f for f in top_features if f in features.columns]

fig, ax = plt.subplots(figsize=(12, 10))
corr_matrix = features[top_features_available].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='RdBu_r', center=0,
            annot=True, fmt='.2f', square=True, linewidths=0.5,
            ax=ax, cbar_kws={'label': 'Pearson r'})
ax.set_title('Correlation Between Top Toxicity-Related Descriptors',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/descriptor_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Pairwise Scatter Plots of Key Descriptors

In [ ]:
# Pairplot of selected descriptors colored by toxicity
plot_cols = ['MolWt', 'MolLogP', 'TPSA', 'NumHDonors']
plot_cols = [c for c in plot_cols if c in features.columns]

plot_df = features[plot_cols].copy()
plot_df['Toxicity'] = pd.Categorical(
    np.where(y == 1, 'Toxic', 'Non-toxic'),
    categories=['Non-toxic', 'Toxic']
)

g = sns.pairplot(plot_df, hue='Toxicity', diag_kind='kde',
                 palette={'Non-toxic': '#2ecc71', 'Toxic': '#e74c3c'},
                 plot_kws={'alpha': 0.4, 's': 15},
                 height=2.5)
g.figure.suptitle('Pairwise Relationships of Key Descriptors', y=1.02,
                  fontsize=14, fontweight='bold')
plt.savefig('../reports/pairplot.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Key Insights & Next Steps

### Key Findings
- **Class Imbalance**: All 12 assays show significant imbalance (toxic class is 3-16%)
- **Missing Data**: 7-26% missing per assay (compounds not tested in all assays)
- **Descriptors**: 217 RDKit molecular descriptors computed from SMILES
- **Correlation**: Some descriptors show clear separation between toxic/non-toxic classes

### Next Steps (Day 2)
1. Train RandomForest and XGBoost models (`src/model.py`)
2. Apply SMOTE to handle class imbalance
3. Evaluate with ROC-AUC and F1 (accounts for imbalance better than accuracy)
4. Run SHAP analysis to identify most important descriptors (`src/explainability.py`)